In [4]:
from pathlib import Path
import sys
import torch 

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0,str(PROJECT_ROOT))

from instancepipelineprior import get_data_loaders
from bayesianprior.discretizedtrain import EarthquakeCNN, evaluate_metrics, logits_to_magnitude, magnitude_to_bin
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

checkpoint_path = PROJECT_ROOT / "bayesianprior" / "data" / "best_model_huberandcrosspoint05.pth"

checkpoint= torch.load(checkpoint_path, map_location=device)
model = EarthquakeCNN().to(device)

model.load_state_dict(checkpoint["model_state_dict"])

model.eval()




EarthquakeCNN(
  (features): Sequential(
    (0): Sequential(
      (0): Conv1d(3, 16, kernel_size=(5,), stride=(1,), padding=(2,))
      (1): ReLU()
      (2): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (1): Sequential(
      (0): Conv1d(16, 32, kernel_size=(5,), stride=(1,), padding=(2,))
      (1): ReLU()
      (2): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (2): Sequential(
      (0): Conv1d(32, 64, kernel_size=(5,), stride=(1,), padding=(2,))
      (1): ReLU()
      (2): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (3): Sequential(
      (0): Conv1d(64, 128, kernel_size=(5,), stride=(1,), padding=(2,))
      (1): ReLU()
      (2): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (4): Sequential(
      (0): Conv1d(128, 256, kernel_size=(5,), stride=(1,), padding=(2,))
      (1): ReLU()
      (2): MaxPool1d(kernel_size=2, str

In [2]:
train_loader,val_loader,test_loader = get_data_loaders()


In [3]:
import inspect, bayesianprior.discretizedtrain as dt
src = inspect.getsource(dt)
for i,line in enumerate(src.splitlines()): 
    if any(k in line.lower() for k in ["bucketize","bin_width","bin_edges","class_idx","class_index","cross_entropy","target"]): print(i, line)

16 BIN_WIDTH = 0.1
21     MIN_MAGNITUDE + BIN_WIDTH / 2,
23     BIN_WIDTH,
27 def magnitude_to_bin(targets):
28     target_bins = ((targets - MIN_MAGNITUDE) / BIN_WIDTH).long()
29     target_bins = torch.clamp(target_bins, 0, NUM_BINS - 1)
30     return target_bins
105     def forward(self, predictions, targets):
107         individual_losses = self.huber(predictions,targets)
110         weights = (1.0+ self.beta* torch.clamp(targets - self.threshold,min=0.0)
118 def train_epoch(model, train_loader, huber_loss_function, cross_entropy_loss_function, gamma, optimizer, device):
129     for batch_idx, (data, target) in enumerate(train_loader):
131         target = target.to(device, non_blocking=True)
138         target_bins = magnitude_to_bin(target)
140         huber_loss = huber_loss_function(predictions, target)
141         cross_loss = cross_entropy_loss_function(logits, target_bins)
151         running_mae += torch.abs(predictions - target).sum().item()
152         correct += (torch.a

In [5]:
import torch

smoothed_prior = torch.load(
    "data/smoothed_magnitude_prior.pt",
    weights_only=False
)

In [6]:
smoothed_prior = torch.tensor(
    smoothed_prior,
    dtype=torch.float32
)

In [7]:
def logits_to_magnitude_with_prior(logits,bin_centers,prior_probs,alpha):
    prior_probs = prior_probs.to(logits.device)
    bin_centers = bin_centers.to(logits.device)
    log_prior = torch.log(prior_probs.clamp(min=1e-8))

    adjusted_logits = (logits + (alpha - 1.0) * log_prior.unsqueeze(0))
    adjusted_probs = torch.softmax(adjusted_logits,dim=1)
    predictions = torch.sum(adjusted_probs* bin_centers.unsqueeze(0),dim=1)
    return predictions, adjusted_probs

In [8]:
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

In [9]:
waveforms, magnitudes = next(
    iter(val_loader)
)

waveforms = waveforms.to(device)
magnitudes = magnitudes.to(device)

print(waveforms.shape)
print(magnitudes.shape)

torch.Size([128, 3, 300])
torch.Size([128])


In [10]:
MIN_MAGNITUDE = 0.0
MAX_MAGNITUDE = 6.6
BIN_WIDTH = 0.1

BIN_CENTERS = torch.arange(
    MIN_MAGNITUDE + BIN_WIDTH / 2,
    MAX_MAGNITUDE,
    BIN_WIDTH,
    dtype=torch.float32
)

In [11]:
import math
import torch
import torch.nn.functional as F
def get_normalized_entropy(logits):

    probs = F.softmax(logits, dim=1)

    entropy = -(probs * torch.log(probs + 1e-12)).sum(dim=1)

    entropy = entropy / math.log(probs.shape[1])

    return entropy


def apply_entropy_prior(logits,prior,bin_centers,alpha_max=0.4,power=1.0,magnitude_threshold=4.0):

    probs = F.softmax(logits, dim=1)

    log_model = torch.log(probs + 1e-12)

    prior = prior.to(device=logits.device,dtype=logits.dtype)

    prior = prior / prior.sum()

    log_prior = torch.log(prior + 1e-12).unsqueeze(0)
    entropy = get_normalized_entropy(logits)

    tail_mask = bin_centers >= magnitude_threshold
    tail_probability = probs[:,tail_mask].sum(dim=1)

    alpha = alpha_max * (entropy ** power) * (1 - tail_probability)
    alpha = alpha.unsqueeze(1)

    log_posterior = ((1 - alpha) * log_model+alpha * log_prior)

    posterior = F.softmax(log_posterior,dim=1)

    return (posterior,entropy,alpha.squeeze(1),tail_probability)

In [12]:
logits = model(waveforms)

posterior, entropy, alpha = apply_entropy_prior(logits,smoothed_prior,alpha_max=0.4,power=1.0)

bin_centers_tensor = torch.tensor(BIN_CENTERS,dtype=posterior.dtype,device=posterior.device)

predictions = (posterior * bin_centers_tensor.unsqueeze(0)).sum(dim=1)

probs = posterior

TypeError: apply_entropy_prior() missing 1 required positional argument: 'bin_centers'

In [ ]:
predictions, probs 

In [ ]:
original_predictions = logits_to_magnitude(
    logits,
    BIN_CENTERS.to(device)
)

print(
    torch.max(
        torch.abs(
            original_predictions - predictions
        )
    )
)

print(
    torch.allclose(
        original_predictions,
        predictions,
        atol=1e-6
    )
)

In [ ]:
import pandas as pd

In [ ]:
def evaluate_metrics_with_prior(model, dataloader, device, prior_probs, alpha_max=0.4, power=1.0, magnitude_threshold=4.0):
    model.eval()

    all_predictions = []
    all_targets = []
    all_entropies = []
    all_alphas = []
    all_tail_probabilities = []

    bin_centers = BIN_CENTERS.to(device)

    prior_probs = torch.as_tensor(
        prior_probs,
        dtype=torch.float32,
        device=device
    )

    with torch.no_grad():
        for waveforms, magnitudes in dataloader:
            waveforms = waveforms.to(device, non_blocking=True)
            magnitudes = magnitudes.to(device, non_blocking=True)

            logits = model(waveforms)

            posterior, entropy, alpha, tail_probability = apply_entropy_prior(
                logits,
                prior_probs,
                bin_centers,
                alpha_max=alpha_max,
                power=power,
                magnitude_threshold=magnitude_threshold
            )

            predictions = (
                posterior * bin_centers.unsqueeze(0)
            ).sum(dim=1)

            all_predictions.append(predictions.cpu())
            all_targets.append(magnitudes.cpu())
            all_entropies.append(entropy.cpu())
            all_alphas.append(alpha.cpu())
            all_tail_probabilities.append(tail_probability.cpu())

    all_predictions = torch.cat(all_predictions, dim=0)
    all_targets = torch.cat(all_targets, dim=0)
    all_entropies = torch.cat(all_entropies, dim=0)
    all_alphas = torch.cat(all_alphas, dim=0)
    all_tail_probabilities = torch.cat(all_tail_probabilities, dim=0)

    errors = all_predictions - all_targets

    mae = torch.abs(errors).mean().item()
    rmse = torch.sqrt(torch.mean(errors ** 2)).item()
    bias = errors.mean().item()
    accuracy = (torch.abs(errors) <= 0.2).float().mean().item() * 100

    results = pd.DataFrame({
        "true_magnitude": all_targets.numpy(),
        "predicted_magnitude": all_predictions.numpy(),
        "entropy": all_entropies.numpy(),
        "tail_probability": all_tail_probabilities.numpy(),
        "alpha": all_alphas.numpy()
    })

    return results, mae, rmse, bias, accuracy

In [ ]:
def evaluate_metrics_with_prior2(model, dataloader, device, prior_probs, alpha_max=0.4, power=1.0):
    model.eval()

    all_predictions = []
    all_base_predictions = []
    all_targets = []
    all_entropies = []
    all_alphas = []

    bin_centers = BIN_CENTERS.to(device)

    prior_probs = torch.as_tensor(
        prior_probs,
        dtype=torch.float32,
        device=device
    )

    with torch.no_grad():
        for waveforms, magnitudes in dataloader:
            waveforms = waveforms.to(device, non_blocking=True)
            magnitudes = magnitudes.to(device, non_blocking=True)

            logits = model(waveforms)

            # Prediction BEFORE applying prior
            probs = torch.softmax(logits, dim=1)

            base_predictions = (
                probs * bin_centers.unsqueeze(0)
            ).sum(dim=1)

            # Apply entropy-gated prior
            posterior, entropy, alpha = apply_entropy_prior(
                logits,
                prior_probs,
                alpha_max=alpha_max,
                power=power
            )

            # Prediction AFTER applying prior
            predictions = (
                posterior * bin_centers.unsqueeze(0)
            ).sum(dim=1)

            all_predictions.append(predictions.cpu())
            all_base_predictions.append(base_predictions.cpu())
            all_targets.append(magnitudes.cpu())
            all_entropies.append(entropy.cpu())
            all_alphas.append(alpha.cpu())

    all_predictions = torch.cat(all_predictions, dim=0)
    all_base_predictions = torch.cat(all_base_predictions, dim=0)
    all_targets = torch.cat(all_targets, dim=0)
    all_entropies = torch.cat(all_entropies, dim=0)
    all_alphas = torch.cat(all_alphas, dim=0)

    errors = all_predictions - all_targets

    mae = torch.abs(errors).mean().item()
    rmse = torch.sqrt(torch.mean(errors ** 2)).item()
    bias = errors.mean().item()
    accuracy = (torch.abs(errors) <= 0.2).float().mean().item() * 100

    results = pd.DataFrame({
        "true_magnitude": all_targets.numpy(),

        "base_prediction": all_base_predictions.numpy(),
        "prior_prediction": all_predictions.numpy(),

        "entropy": all_entropies.numpy(),
        "alpha": all_alphas.numpy()
    })

    # Error BEFORE prior
    results["base_error"] = (
        results["base_prediction"]
        - results["true_magnitude"]
    )

    results["base_abs_error"] = (
        results["base_error"].abs()
    )

    # Error AFTER prior
    results["prior_error"] = (
        results["prior_prediction"]
        - results["true_magnitude"]
    )

    results["prior_abs_error"] = (
        results["prior_error"].abs()
    )

    return results, mae, rmse, bias, accuracy

In [ ]:
import pandas as pd

In [ ]:
results, mae, rmse, bias, accuracy = evaluate_metrics_with_prior2(model,val_loader,
    device,
    smoothed_prior,
    alpha_max=0.4,
    power=1.0
)

In [ ]:
from scipy.stats import spearmanr

corr, p = spearmanr(
    results["entropy"],
    results["base_abs_error"]
)

print("Spearman correlation:", corr)
print("p-value:", p)

Spearman correlation: 0.33903464687421353
p-value: 0.0


In [ ]:
bins = [0, 1, 2, 3, 4, 5, 6]

results["magnitude_bin"] = pd.cut(
    results["true_magnitude"],
    bins=bins,
    right=False
)

magnitude_summary = results.groupby(
    "magnitude_bin",
    observed=True
).agg(
    count=("true_magnitude", "size"),
    mean_entropy=("entropy", "mean"),
    base_mae=("base_abs_error", "mean"),
    base_bias=("base_error", "mean")
)

print(magnitude_summary)

               count  mean_entropy  base_mae  base_bias
magnitude_bin                                          
[0, 1)          5190      0.682420  0.727607   0.727607
[1, 2)         20718      0.672188  0.422286   0.374928
[2, 3)         60450      0.654695  0.275434   0.016304
[3, 4)          9608      0.705847  0.536290  -0.360271
[4, 5)           936      0.735259  0.695998  -0.609000
[5, 6)            91      0.767078  1.015091  -0.985859


In [ ]:
results.keys()

Index(['true_magnitude', 'base_prediction', 'prior_prediction', 'entropy',
       'alpha', 'base_error', 'base_abs_error', 'prior_error',
       'prior_abs_error'],
      dtype='object')

In [ ]:
results.to_csv("resultsentropy3s.csv")

In [ ]:
results2, mae, rmse, bias, accuracy = evaluate_metrics_with_prior(
    model,
    val_loader,
    device,
    smoothed_prior,
    alpha_max=0.4,
    power=1.0,
    magnitude_threshold=4.0
)

In [ ]:
print("MAE:", mae)
print("RMSE:", rmse)
print("Bias:", bias)
print("Accuracy ±0.2:", accuracy)

MAE: 0.36415040493011475
RMSE: 0.4876316487789154
Bias: 0.10071167349815369
Accuracy ±0.2: 38.39142918586731


In [ ]:
bins = [0, 1, 2, 3, 4, 5, 6]

results2["magnitude_bin"] = pd.cut(
    results2["true_magnitude"],
    bins=bins,
    right=False
)

results2["error"] = (
    results2["predicted_magnitude"]
    - results2["true_magnitude"]
)

results2["abs_error"] = results2["error"].abs()

summary = results2.groupby(
    "magnitude_bin",
    observed=True
).agg(
    count=("true_magnitude", "size"),
    mae=("abs_error", "mean"),
    bias=("error", "mean"),
    mean_entropy=("entropy", "mean"),
    mean_tail_probability=("tail_probability", "mean"),
    mean_alpha=("alpha", "mean")
)

print(summary)

               count       mae      bias  mean_entropy  mean_tail_probability  \
magnitude_bin                                                                   
[0, 1)          5190  0.837370  0.837370      0.682420               0.000983   
[1, 2)         20718  0.475781  0.452594      0.672188               0.001820   
[2, 3)         60450  0.248078  0.016564      0.654695               0.011248   
[3, 4)          9608  0.550655 -0.434024      0.705847               0.098766   
[4, 5)           936  0.773018 -0.725789      0.735259               0.423964   
[5, 6)            91  1.168247 -1.168247      0.767078               0.525048   

               mean_alpha  
magnitude_bin              
[0, 1)           0.272662  
[1, 2)           0.268313  
[2, 3)           0.258486  
[3, 4)           0.252980  
[4, 5)           0.169831  
[5, 6)           0.143204  


In [ ]:
print(results2[
    ["entropy", "tail_probability", "alpha"]
].describe())

            entropy  tail_probability         alpha
count  96993.000000      96993.000000  96993.000000
mean       0.665865          0.021819      0.259834
std        0.060258          0.091378      0.029894
min        0.014801          0.000000      0.005920
25%        0.627049          0.000004      0.248041
50%        0.665613          0.000215      0.263310
75%        0.708495          0.003146      0.277159
max        0.910969          0.973763      0.323299


In [ ]:
results2["prior_strength_reduction"] = (
    results2["entropy"] * 0.4
    - results2["alpha"]
)

In [ ]:
print(
    results2.groupby(
        "magnitude_bin",
        observed=True
    )["prior_strength_reduction"].mean()
)

magnitude_bin
[0, 1)    0.000306
[1, 2)    0.000563
[2, 3)    0.003393
[3, 4)    0.029359
[4, 5)    0.124273
[5, 6)    0.163627
Name: prior_strength_reduction, dtype: float32


In [ ]:
results2.to_csv("detailedentropytail3s.csv")

If I remove some of the training magnitude-frequency bias from the existing classifier logits, does the rare-large-earthquake underestimation improve?

In [8]:
MIN_MAGNITUDE = 0.0
MAX_MAGNITUDE = 6.6
BIN_WIDTH = 0.1

BIN_EDGES = torch.arange(
    MIN_MAGNITUDE,
    MAX_MAGNITUDE + BIN_WIDTH,
    BIN_WIDTH
)

BIN_CENTERS = (
    BIN_EDGES[:-1] + BIN_EDGES[1:]
) / 2

In [5]:
def compute_empirical_prior(train_loader):
    counts = torch.zeros(len(BIN_CENTERS), dtype=torch.float64)
    for _, magnitudes in train_loader:
        magnitudes = magnitudes.cpu()
        magnitudes = magnitudes[(magnitudes >= MIN_MAGNITUDE) & (magnitudes < MAX_MAGNITUDE)]
        indices = magnitude_to_bin(magnitudes)
        counts += torch.bincount(indices, minlength=len(BIN_CENTERS))
    prior = counts / counts.sum()
    return counts.float(), prior.float()

In [9]:
train_counts, empirical_prior = compute_empirical_prior(train_loader)

In [10]:
zero_bins = torch.where(train_counts == 0)[0]

In [11]:
print("Number of zero bins:", len(zero_bins))

Number of zero bins: 11


In [12]:
for i in zero_bins: print(f"index={i.item():2d}  center={BIN_CENTERS[i].item():.2f}  count={int(train_counts[i])}")

index=13  center=1.35  count=0
index=21  center=2.15  count=0
index=26  center=2.65  count=0
index=31  center=3.15  count=0
index=42  center=4.25  count=0
index=47  center=4.75  count=0
index=52  center=5.25  count=0
index=57  center=5.75  count=0
index=62  center=6.25  count=0
index=63  center=6.35  count=0
index=64  center=6.45  count=0


In [13]:
for i,(center,count) in enumerate(zip(BIN_CENTERS,train_counts)): print(f"{i:2d}  M={center:.2f}  count={int(count)}")

 0  M=0.05  count=24
 1  M=0.15  count=71
 2  M=0.25  count=228
 3  M=0.35  count=467
 4  M=0.45  count=1363
 5  M=0.55  count=2076
 6  M=0.65  count=2804
 7  M=0.75  count=5058
 8  M=0.85  count=7336
 9  M=0.95  count=10734
10  M=1.05  count=13779
11  M=1.15  count=15506
12  M=1.25  count=39295
13  M=1.35  count=0
14  M=1.45  count=19406
15  M=1.55  count=18871
16  M=1.65  count=17984
17  M=1.75  count=15328
18  M=1.85  count=13631
19  M=1.95  count=12135
20  M=2.05  count=206013
21  M=2.15  count=0
22  M=2.25  count=88570
23  M=2.35  count=72826
24  M=2.45  count=64785
25  M=2.55  count=108932
26  M=2.65  count=0
27  M=2.75  count=41562
28  M=2.85  count=34526
29  M=2.95  count=28383
30  M=3.05  count=45882
31  M=3.15  count=0
32  M=3.25  count=17823
33  M=3.35  count=14508
34  M=3.45  count=11345
35  M=3.55  count=9895
36  M=3.65  count=5703
37  M=3.75  count=6226
38  M=3.85  count=3927
39  M=3.95  count=4082
40  M=4.05  count=4189
41  M=4.15  count=4960
42  M=4.25  count=0
43  M=4.

In [15]:
train_counts, empirical_prior = compute_empirical_prior(
    train_loader
)

print(train_counts)
print(empirical_prior)

tensor([2.4000e+01, 7.1000e+01, 2.2800e+02, 4.6700e+02, 1.3630e+03, 2.0760e+03,
        2.8040e+03, 5.0580e+03, 1.8070e+04, 1.0000e+00, 1.3779e+04, 1.5506e+04,
        1.9620e+04, 1.9675e+04, 1.9406e+04, 1.8871e+04, 1.7984e+04, 2.8959e+04,
        1.0000e+00, 1.2135e+04, 1.0962e+05, 9.6397e+04, 8.8570e+04, 7.2826e+04,
        6.4785e+04, 1.0893e+05, 1.0000e+00, 7.6088e+04, 1.0000e+00, 2.8383e+04,
        4.5882e+04, 1.0000e+00, 1.7823e+04, 1.4508e+04, 1.1345e+04, 1.5598e+04,
        1.0000e+00, 6.2260e+03, 3.9270e+03, 4.0820e+03, 4.1890e+03, 2.4210e+03,
        2.5390e+03, 2.1560e+03, 1.6260e+03, 1.4390e+03, 6.1600e+02, 5.9800e+02,
        6.8700e+02, 4.6600e+02, 5.9800e+02, 9.7000e+01, 1.0000e+00, 4.8000e+01,
        3.2200e+02, 4.3000e+01, 4.5000e+01, 1.0000e+00, 2.7700e+02, 7.2000e+01,
        3.9000e+01, 5.0000e+01, 1.0000e+00, 1.0000e+00, 1.0000e+00, 7.5000e+01])
tensor([2.4502e-05, 7.2486e-05, 2.3277e-04, 4.7677e-04, 1.3915e-03, 2.1195e-03,
        2.8627e-03, 5.1639e-03, 1.8448e

In [17]:
def collect_logits(model,dataloader,device):
    model.eval(); all_logits,all_targets=[],[]
    with torch.no_grad():
        for waveforms,magnitudes in dataloader:
            waveforms=waveforms.to(device,non_blocking=True); logits=model(waveforms)
            all_logits.append(logits.cpu()); all_targets.append(magnitudes.cpu())
    return torch.cat(all_logits),torch.cat(all_targets)

In [18]:
val_logits,val_targets=collect_logits(model,val_loader,device)

In [14]:
different, total = 0, 0

In [15]:
for _,m in train_loader:
    m=m.cpu(); m=m[(m>=MIN_MAGNITUDE)&(m<MAX_MAGNITUDE)]; old=torch.bucketize(m,BIN_EDGES[1:-1],right=True); actual=magnitude_to_bin(m); different+=(old!=actual).sum().item(); total+=len(m)

In [16]:
print(f"Different class assignments: {different}/{total} = {100*different/total:.3f}%")

Different class assignments: 183948/979487 = 18.780%


In [19]:
zero_mask=empirical_prior==0
base_probs=torch.softmax(val_logits,dim=1)
print("Mean probability mass on zero bins:",base_probs[:,zero_mask].sum(dim=1).mean().item())
print("Argmax in zero bin %:",zero_mask[val_logits.argmax(dim=1)].float().mean().item()*100)

Mean probability mass on zero bins: 0.189993217587471
Argmax in zero bin %: 1.9939582794904709


In [20]:
masked_logits = val_logits.clone(); masked_logits[:,zero_mask] = -torch.inf

In [21]:
base_predictions = (torch.softmax(val_logits,dim=1) * BIN_CENTERS.unsqueeze(0)).sum(dim=1)

In [22]:
masked_predictions = (torch.softmax(masked_logits,dim=1) * BIN_CENTERS.unsqueeze(0)).sum(dim=1)

In [23]:
def calculate_metrics(predictions, targets): errors=predictions-targets; return {"mae":torch.abs(errors).mean().item(),"rmse":torch.sqrt(torch.mean(errors**2)).item(),"bias":errors.mean().item(),"accuracy_0.2":(torch.abs(errors)<=0.2).float().mean().item()*100,"underestimate_rate":(predictions<targets).float().mean().item()*100,"underestimate_0.5_rate":(predictions<targets-0.5).float().mean().item()*100}

In [24]:
print("RAW:",calculate_metrics(base_predictions,val_targets))
print("MASKED:",calculate_metrics(masked_predictions,val_targets))

RAW: {'mae': 0.361589640378952, 'rmse': 0.48687827587127686, 'bias': 0.08669096976518631, 'accuracy_0.2': 38.67392539978027, 'underestimate_rate': 41.107091307640076, 'underestimate_0.5_rate': 9.11509096622467}
MASKED: {'mae': 0.36139172315597534, 'rmse': 0.48556411266326904, 'bias': 0.08050640672445297, 'accuracy_0.2': 38.502779603004456, 'underestimate_rate': 41.48443639278412, 'underestimate_0.5_rate': 9.375934302806854}


In [25]:
def tail_metrics(predictions,targets,threshold): mask=targets>=threshold; p,y=predictions[mask],targets[mask]; e=p-y; return {"count":len(y),"mae":torch.abs(e).mean().item(),"bias":e.mean().item(),"underestimate_rate":(p<y).float().mean().item()*100,"underestimate_0.5_rate":(p<=y-0.5).float().mean().item()*100}

In [26]:
print("RAW M>=3.5:",tail_metrics(base_predictions,val_targets,3.5))
print("MASKED M>=3.5:",tail_metrics(masked_predictions,val_targets,3.5))
print("RAW M>=4.0:",tail_metrics(base_predictions,val_targets,4.0))
print("MASKED M>=4.0:",tail_metrics(masked_predictions,val_targets,4.0))

RAW M>=3.5: {'count': 3266, 'mae': 0.674118161201477, 'bias': -0.5338048934936523, 'underestimate_rate': 77.21983790397644, 'underestimate_0.5_rate': 50.765460729599}
MASKED M>=3.5: {'count': 3266, 'mae': 0.6605489253997803, 'bias': -0.5308777689933777, 'underestimate_rate': 77.6485025882721, 'underestimate_0.5_rate': 49.75505173206329}
RAW M>=4.0: {'count': 1027, 'mae': 0.7242725491523743, 'bias': -0.6423926949501038, 'underestimate_rate': 82.96007513999939, 'underestimate_0.5_rate': 48.880234360694885}
MASKED M>=4.0: {'count': 1027, 'mae': 0.7128558158874512, 'bias': -0.6597651243209839, 'underestimate_rate': 85.68646311759949, 'underestimate_0.5_rate': 48.39337766170502}


In [30]:
def magnitude_to_bin_fixed(targets): return torch.floor((targets - MIN_MAGNITUDE) / BIN_WIDTH + 1e-5).long().clamp(0, NUM_BINS - 1)

In [29]:
NUM_BINS = len(BIN_CENTERS)

In [31]:
fixed_counts = torch.zeros(NUM_BINS,dtype=torch.long)

In [32]:
for _,m in train_loader:
    m=m.cpu(); idx=magnitude_to_bin_fixed(m); fixed_counts += torch.bincount(idx,minlength=NUM_BINS)

In [33]:
fixed_zero_bins = torch.where(fixed_counts == 0)[0]
print("Number of zero bins:",len(fixed_zero_bins))
for i in fixed_zero_bins: print(f"index={i.item()}  center={BIN_CENTERS[i].item():.2f}  count={fixed_counts[i].item()}")

Number of zero bins: 4
index=57  center=5.75  count=0
index=62  center=6.25  count=0
index=63  center=6.35  count=0
index=64  center=6.45  count=0


In [34]:
for i,(center,count) in enumerate(zip(BIN_CENTERS,fixed_counts)): print(f"{i:2d}  M={center:.2f}  count={int(count)}")

 0  M=0.05  count=24
 1  M=0.15  count=71
 2  M=0.25  count=228
 3  M=0.35  count=467
 4  M=0.45  count=1363
 5  M=0.55  count=2076
 6  M=0.65  count=2804
 7  M=0.75  count=5058
 8  M=0.85  count=7336
 9  M=0.95  count=10734
10  M=1.05  count=13779
11  M=1.15  count=15506
12  M=1.25  count=19620
13  M=1.35  count=19675
14  M=1.45  count=19406
15  M=1.55  count=18871
16  M=1.65  count=17984
17  M=1.75  count=15328
18  M=1.85  count=13631
19  M=1.95  count=12135
20  M=2.05  count=109616
21  M=2.15  count=96397
22  M=2.25  count=88570
23  M=2.35  count=72826
24  M=2.45  count=64785
25  M=2.55  count=59954
26  M=2.65  count=48978
27  M=2.75  count=41562
28  M=2.85  count=34526
29  M=2.95  count=28383
30  M=3.05  count=25092
31  M=3.15  count=20790
32  M=3.25  count=17823
33  M=3.35  count=14508
34  M=3.45  count=11345
35  M=3.55  count=9895
36  M=3.65  count=5703
37  M=3.75  count=6226
38  M=3.85  count=3927
39  M=3.95  count=4082
40  M=4.05  count=4189
41  M=4.15  count=2421
42  M=4.25  c

In [35]:
import importlib, bayesianprior.discretizedtrain as dt
importlib.reload(dt)

<module 'bayesianprior.discretizedtrain' from '/home/ec2-user/Earthquake/bayesianprior/discretizedtrain.py'>

In [36]:
import inspect
print(inspect.getsource(dt.magnitude_to_bin))

def magnitude_to_bin(targets):
    target_bins = torch.floor((targets - MIN_MAGNITUDE) / BIN_WIDTH + 1e-5).long()
    target_bins = torch.clamp(target_bins, 0, NUM_BINS - 1)
    return target_bins



In [37]:
test_mags=torch.tensor([1.3,2.1,2.6,3.1,4.2,4.7,5.2])
print(test_mags)
print(dt.magnitude_to_bin(test_mags))

tensor([1.3000, 2.1000, 2.6000, 3.1000, 4.2000, 4.7000, 5.2000])
tensor([13, 21, 26, 31, 42, 47, 52])


In [16]:
def collect_logits(model, dataloader, device):
    model.eval()

    all_logits = []
    all_targets = []

    with torch.no_grad():
        for waveforms, magnitudes in dataloader:

            waveforms = waveforms.to(device,non_blocking=True)

            logits = model(waveforms)

            all_logits.append(logits.cpu())

            all_targets.append(magnitudes.cpu())

    return (torch.cat(all_logits),torch.cat(all_targets))

In [17]:
val_logits, val_targets = collect_logits(
    model,
    val_loader,
    device
)

print(val_logits.shape)
print(val_targets.shape)

torch.Size([96993, 66])
torch.Size([96993])


In [18]:
assert val_logits.shape[1] == len(BIN_CENTERS)

In [19]:
def apply_logit_adjustment(logits,prior,tau):
    prior = prior.to(device=logits.device,dtype=logits.dtype)

    log_prior = torch.log(prior.clamp_min(1e-12))

    adjusted_logits = (logits- tau * log_prior.unsqueeze(0))
    return adjusted_logits

In [20]:
def predict_with_logit_adjustment(logits,prior,bin_centers,tau):
    adjusted_logits = apply_logit_adjustment(logits,prior,tau)

    probs = torch.softmax(adjusted_logits,dim=1)

    bin_centers = bin_centers.to( probs.device)

    expected_prediction = (probs* bin_centers.unsqueeze(0)).sum(dim=1)

    argmax_prediction = bin_centers[probs.argmax(dim=1)]

    return (expected_prediction,argmax_prediction,probs,adjusted_logits)

In [21]:
pred, argmax_pred, probs, adjusted_logits = predict_with_logit_adjustment(val_logits,empirical_prior,BIN_CENTERS,tau=0.0)


In [22]:
base_probs = torch.softmax(
    val_logits,
    dim=1
)

base_predictions = (
    base_probs
    * BIN_CENTERS.unsqueeze(0)
).sum(dim=1)

print(
    torch.max(
        torch.abs(
            base_predictions - pred
        )
    )
)

print(
    torch.allclose(
        base_predictions,
        pred,
        atol=1e-6
    )
)

tensor(0.)
True


In [23]:
taus = [0.0, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 2.0]

In [24]:
def calculate_metrics(predictions, targets):
    errors = predictions - targets
    return {"mae": torch.abs(errors).mean().item(), "rmse": torch.sqrt(torch.mean(errors ** 2)).item(), "bias": errors.mean().item(), "accuracy_0.2": (torch.abs(errors) <= 0.2).float().mean().item() * 100, "underestimate_rate": (predictions < targets).float().mean().item() * 100, "underestimate_0.5_rate": (predictions < targets - 0.5).float().mean().item() * 100}

In [25]:
def evaluate_tau(logits, targets, prior, tau):
    predictions, argmax_predictions, probs, _ = predict_with_logit_adjustment(logits, prior, BIN_CENTERS, tau)
    overall = calculate_metrics(predictions, targets)
    df = pd.DataFrame({"true_magnitude": targets.numpy(), "prediction": predictions.numpy(), "argmax_prediction": argmax_predictions.numpy()})
    df["error"] = df["prediction"] - df["true_magnitude"]
    df["abs_error"] = df["error"].abs()
    df["magnitude_bin"] = pd.cut(df["true_magnitude"], bins=[0, 1, 2, 3, 4, 5, 6], right=False)
    bin_summary = df.groupby("magnitude_bin", observed=True).agg(count=("true_magnitude", "size"), mae=("abs_error", "mean"), bias=("error", "mean"))
    return overall, bin_summary, df

In [26]:
tau_results, bin_results = [], {}

In [28]:
import pandas as pd

In [29]:
for tau in taus:
    overall, bin_summary, df = evaluate_tau(val_logits, val_targets, empirical_prior, tau)
    overall["tau"] = tau
    tau_results.append(overall)
    bin_results[tau] = bin_summary

In [30]:
tau_results = pd.DataFrame(tau_results)

In [35]:
print(tau_results)

        mae      rmse      bias  accuracy_0.2  underestimate_rate  \
0  0.361590  0.486878  0.086691     38.673925           41.107091   
1  0.387082  0.510010  0.157095     33.913788           34.112772   
2  0.410945  0.531599  0.196291     30.497047           31.170291   
3  0.414637  0.535402  0.198800     30.074334           30.880579   
4  0.415180  0.536136  0.198445     30.024847           30.885735   
5  0.415277  0.536287  0.198311     30.011445           30.878517   
6  0.415296  0.536319  0.198281     30.008352           30.880579   
7  0.415301  0.536328  0.198275     30.007321           30.878517   

   underestimate_0.5_rate   tau  
0                9.115091  0.00  
1                7.729424  0.25  
2                7.293310  0.50  
3                7.378883  0.75  
4                7.427340  1.00  
5                7.435588  1.25  
6                7.435588  1.50  
7                7.436619  2.00  


In [32]:
for tau in taus:
    print(f"\nTAU = {tau}")
    print(bin_results[tau])


TAU = 0.0
               count       mae      bias
magnitude_bin                           
[0, 1)          5190  0.727607  0.727607
[1, 2)         20718  0.422286  0.374928
[2, 3)         60450  0.275434  0.016304
[3, 4)          9608  0.536290 -0.360271
[4, 5)           936  0.695998 -0.609000
[5, 6)            91  1.015091 -0.985859

TAU = 0.25
               count       mae      bias
magnitude_bin                           
[0, 1)          5190  0.676746  0.676746
[1, 2)         20718  0.443736  0.363447
[2, 3)         60450  0.327314  0.131027
[3, 4)          9608  0.440521 -0.316414
[4, 5)           936  0.762079 -0.646506
[5, 6)            91  1.171856 -0.883616

TAU = 0.5
               count       mae      bias
magnitude_bin                           
[0, 1)          5190  0.688451  0.688451
[1, 2)         20718  0.480524  0.391297
[2, 3)         60450  0.354282  0.192809
[3, 4)          9608  0.405371 -0.353646
[4, 5)           936  0.956540 -0.860441
[5, 6)            91  1

In [33]:
def balanced_bin_mae(bin_summary): return bin_summary["mae"].mean()

In [36]:
tau_results = []

In [37]:
for tau in taus:
    overall, bin_summary, df = evaluate_tau(val_logits, val_targets, empirical_prior, tau)
    overall["tau"] = tau
    overall["balanced_bin_mae"] = bin_summary["mae"].mean()
    tau_results.append(overall)

In [38]:
tau_results = pd.DataFrame(tau_results)

In [39]:
print(tau_results.sort_values("balanced_bin_mae"))

        mae      rmse      bias  accuracy_0.2  underestimate_rate  \
0  0.361590  0.486878  0.086691     38.673925           41.107091   
1  0.387082  0.510010  0.157095     33.913788           34.112772   
2  0.410945  0.531599  0.196291     30.497047           31.170291   
3  0.414637  0.535402  0.198800     30.074334           30.880579   
4  0.415180  0.536136  0.198445     30.024847           30.885735   
5  0.415277  0.536287  0.198311     30.011445           30.878517   
6  0.415296  0.536319  0.198281     30.008352           30.880579   
7  0.415301  0.536328  0.198275     30.007321           30.878517   

   underestimate_0.5_rate   tau  balanced_bin_mae  
0                9.115091  0.00          0.612118  
1                7.729424  0.25          0.637042  
2                7.293310  0.50          0.707571  
3                7.378883  0.75          0.749595  
4                7.427340  1.00          0.762904  
5                7.435588  1.25          0.766209  
6             

In [40]:
def tail_metrics(predictions, targets, threshold):
    mask = targets >= threshold
    p, y = predictions[mask], targets[mask]
    error = p - y
    return {"count": len(y), "mae": torch.abs(error).mean().item(), "bias": error.mean().item(), "underestimate_rate": (p < y).float().mean().item() * 100, "underestimate_0.5_rate": (p <= y - 0.5).float().mean().item() * 100}

In [41]:
for tau in taus:
    predictions, _, _, _ = predict_with_logit_adjustment(val_logits, empirical_prior, BIN_CENTERS, tau)
    print("\nTau:", tau)
    print("M >= 3.5:", tail_metrics(predictions, val_targets, 3.5))
    print("M >= 4.0:", tail_metrics(predictions, val_targets, 4.0))


Tau: 0.0
M >= 3.5: {'count': 3266, 'mae': 0.674118161201477, 'bias': -0.5338048934936523, 'underestimate_rate': 77.21983790397644, 'underestimate_0.5_rate': 50.765460729599}
M >= 4.0: {'count': 1027, 'mae': 0.7242725491523743, 'bias': -0.6423926949501038, 'underestimate_rate': 82.96007513999939, 'underestimate_0.5_rate': 48.880234360694885}

Tau: 0.25
M >= 3.5: {'count': 3266, 'mae': 0.6670792102813721, 'bias': -0.5474931001663208, 'underestimate_rate': 83.43539237976074, 'underestimate_0.5_rate': 55.419474840164185}
M >= 4.0: {'count': 1027, 'mae': 0.7983880043029785, 'bias': -0.6675158739089966, 'underestimate_rate': 85.68646311759949, 'underestimate_0.5_rate': 57.254135608673096}

Tau: 0.5
M >= 3.5: {'count': 3266, 'mae': 0.7436835169792175, 'bias': -0.6823222041130066, 'underestimate_rate': 93.69258880615234, 'underestimate_0.5_rate': 68.43233108520508}
M >= 4.0: {'count': 1027, 'mae': 0.9923130869865417, 'bias': -0.8749139308929443, 'underestimate_rate': 92.79454946517944, 'under

In [42]:
def threshold_metrics(predictions, targets, threshold=4.0):
    pred_positive, true_positive = predictions >= threshold, targets >= threshold
    tp = (pred_positive & true_positive).sum().item()
    fp = (pred_positive & ~true_positive).sum().item()
    fn = (~pred_positive & true_positive).sum().item()
    precision = tp / (tp + fp) if tp + fp > 0 else 0
    recall = tp / (tp + fn) if tp + fn > 0 else 0
    return {"TP": tp, "FP": fp, "FN": fn, "precision": precision, "recall": recall}

In [43]:
for tau in taus:
    predictions, _, _, _ = predict_with_logit_adjustment(val_logits, empirical_prior, BIN_CENTERS, tau)
    print(tau, threshold_metrics(predictions, val_targets, 4.0))

0.0 {'TP': 435, 'FP': 572, 'FN': 592, 'precision': 0.43197616683217477, 'recall': 0.42356377799415773}
0.25 {'TP': 356, 'FP': 354, 'FN': 671, 'precision': 0.5014084507042254, 'recall': 0.3466407010710808}
0.5 {'TP': 143, 'FP': 138, 'FN': 884, 'precision': 0.5088967971530249, 'recall': 0.13924050632911392}
0.75 {'TP': 79, 'FP': 83, 'FN': 948, 'precision': 0.4876543209876543, 'recall': 0.07692307692307693}
1.0 {'TP': 74, 'FP': 80, 'FN': 953, 'precision': 0.4805194805194805, 'recall': 0.07205452775073028}
1.25 {'TP': 74, 'FP': 79, 'FN': 953, 'precision': 0.48366013071895425, 'recall': 0.07205452775073028}
1.5 {'TP': 74, 'FP': 79, 'FN': 953, 'precision': 0.48366013071895425, 'recall': 0.07205452775073028}
2.0 {'TP': 74, 'FP': 79, 'FN': 953, 'precision': 0.48366013071895425, 'recall': 0.07205452775073028}
